# Fine-tune Whisper with LoRA

Whisper turns the worker's speech into text. Here I fine-tune **Whisper small.en** with **LoRA**: Whisper's own weights stay frozen and small extra weights (rank 16, in the attention layers `q_proj` and `v_proj`) are trained, the same idea as my Qwen LoRA.

- **Training audio** (`audio_train/`): the 210 *training* sentences read by 6 synthetic voices, clean and with engine noise (420 clips).
- **Test audio**: the 100 *test* sentences in 4 *other* synthetic voices (`audio/`), and in **my own voice** (`my_recordings/full100/`). My voice is only used for testing.

Runtime: **GPU (T4)**. My results came from the same scripts on my laptop (RTX 3050, 21 minutes of training).

In [ ]:
!git clone -q https://github.com/Yarida-nes/yard-voice-lora.git
%cd yard-voice-lora

In [ ]:
!pip install -q peft jiwer av

## 1. (Optional) Make the training audio again
The 420 training clips are already in `audio_train/`. To rebuild them from `data/train.jsonl` (needs internet for the synthetic voices):

In [ ]:
# !pip install -q edge-tts
# !python optional/make_train_audio.py

## 2. Train LoRA Whisper
4 epochs, learning rate 5e-4, batch 8. Saves the LoRA weights to `models/whisper-lora`. My run: 1.77 million trainable weights (0.73%), loss 0.57 → 0.04.

In [ ]:
!python optional/train_whisper_lora.py

## 3. Test base vs LoRA Whisper
Both versions transcribe the same clips. It prints how many trailer numbers were heard right and the word error rate.

My results:

| | Synthetic test voices (clean / noisy) | My own voice (100 sentences) |
|---|---|---|
| Base Whisper | 83 / 80 | 41 |
| LoRA Whisper | 93 / 93 | 60 |

In [ ]:
!python optional/transcribe_lora.py                         # synthetic test voices (audio/)
!python optional/transcribe_lora.py my_recordings/full100   # my own voice

## 4. Listen to an example
What I said, what base Whisper heard, and what LoRA Whisper heard.

In [ ]:
import json
from IPython.display import Audio, display
b = {r['id']: r for r in map(json.loads, open('data/my_recordings_full100_clean_wbase.jsonl'))}
l = {r['id']: r for r in map(json.loads, open('data/my_recordings_full100_clean_wlora.jsonl'))}
i = 'test-0033'
print('I said:      ', b[i]['original_text'])
print('Base heard:  ', b[i]['text'])
print('LoRA heard:  ', l[i]['text'])
display(Audio('my_recordings/full100/' + b[i]['audio']))